# 03 — Build Custom Judges for Agent Evaluation
- Date: 9-17-2026

This notebook shows how to build **custom scorers** when the built-in judges don’t cover
your specific evaluation needs.

You’ll learn:
* How to write a **code-based scorer** with the `@scorer` decorator (deterministic, zero latency)
* How to build a **custom LLM judge** with `make_judge()` (flexible, domain-specific)
* How to combine custom + built-in scorers in a single evaluation run

**When to use custom judges:**
* Domain-specific rules (e.g., “agent must cite order IDs in response”)
* Deterministic checks that don’t need an LLM (e.g., “response contains a tracking number”)
* Remote/managed agents where TOOL spans aren’t visible

**Compute:** DBR 18.2 ML cluster (`mlflow-eval-suite`)  
**Dependencies:** All pre-installed — no `%pip install` required.

**Prerequisites:** Run notebooks 01 and 02 first.

In [0]:
# No installs needed — DBR 18.2 ML ships with mlflow 3.x, openai, and databricks-agents.
import mlflow
print(f"MLflow {mlflow.__version__} — ready to go.")

MLflow 3.8.1 — ready to go.


In [0]:
"""Setup: same agent as notebooks 01/02 + import scorer utilities."""
import openai
import json
import pandas as pd
from mlflow.entities import SpanType
from mlflow.genai.scorers import scorer, Guidelines, ToolCallEfficiency
from mlflow.genai.judges import make_judge

# Enable OpenAI autologging
mlflow.openai.autolog()

_username = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
)
EXPERIMENT_NAME = f"/Users/{_username}/agentic-evals-intro"
mlflow.set_experiment(EXPERIMENT_NAME)

# --- Tool definition ---
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up the status of a customer order by order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID (e.g. ORD-1001)"}
                },
                "required": ["order_id"],
            },
        },
    }
]

## mock dictionary
def lookup_order(order_id: str) -> str:
    """Look up the status of a customer order by order ID."""
    orders = {
        "ORD-1001": "Shipped \u2014 arrives Thursday",
        "ORD-1002": "Processing \u2014 payment confirmed, preparing for shipment",
        "ORD-1003": "Delivered \u2014 left at front door on Monday",
        "ORD-1004": "Cancelled \u2014 refund issued",
    }
    return orders.get(order_id, f"Order {order_id} not found in system")

# --- Agent ---
MODEL_ENDPOINT = "databricks-claude-sonnet-4"
TOOL_FUNCTIONS = {"lookup_order": lookup_order}

client = openai.OpenAI(
    base_url=(
        f"https://{dbutils.notebook.entry_point.getDbutils().notebook().getContext().browserHostName().get()}"
        "/serving-endpoints"
    ),
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
)


@mlflow.trace(span_type=SpanType.AGENT)
def run_agent(user_message: str) -> str:
    """Simple tool-calling agent: LLM → tool execution → LLM."""
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model=MODEL_ENDPOINT, messages=messages, tools=TOOLS
    )
    assistant_msg = response.choices[0].message

    if assistant_msg.tool_calls:
        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in assistant_msg.tool_calls
            ],
        })

        for tool_call in assistant_msg.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            with mlflow.start_span(name=fn_name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                result = TOOL_FUNCTIONS[fn_name](**args)
                span.set_outputs({"result": result})

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

        response = client.chat.completions.create(model=MODEL_ENDPOINT, messages=messages)
        return response.choices[0].message.content

    return assistant_msg.content


print(f"\u2713 Agent ready (LLM: {MODEL_ENDPOINT}, Tools: [lookup_order])")

✓ Agent ready (LLM: databricks-claude-sonnet-4, Tools: [lookup_order])


## Code-Based Scorers with `@scorer`
- The simplest custom judge is a **code-based scorer** — a Python function decorated with `@scorer`.
- **It runs deterministically (no LLM call), so it’s fast, cheap, and 100% reproducible.**

Use cases:
* Check if the response contains specific keywords or patterns
* Validate response format (JSON, markdown, etc.)
* Count tool calls or verify specific tool arguments
* Any rule you can express in Python code

In [0]:
"""Define code-based scorers using the @scorer decorator."""


@scorer
def mentions_order_id(*, inputs, outputs) -> bool:
    """Check if the response mentions a specific order ID when one was asked about."""
    import re

    # Extract user question
    messages = inputs.get("messages", [])
    user_msg = messages[0]["content"] if messages else ""

    # If the question mentions an order ID, the response should too
    order_ids_in_question = re.findall(r"ORD-\d+", user_msg)

    if not order_ids_in_question:
        return True  # No order ID in question, so no requirement

    # Check if at least one order ID appears in the response
    return any(oid in str(outputs) for oid in order_ids_in_question)


@scorer
def response_length(*, outputs) -> int:
    """Return the response length in characters (for tracking, not pass/fail)."""
    return len(str(outputs)) if outputs else 0


@scorer
def no_hallucinated_status(*, outputs) -> bool:
    """Ensure the response doesn't invent order statuses not in our system."""
    valid_statuses = ["shipped", "processing", "delivered", "cancelled", "not found"]
    response_lower = str(outputs).lower()

    # If it mentions a status-like word, it should be one of our known statuses
    status_words = ["shipped", "processing", "delivered", "cancelled",
                    "pending", "returned", "refunded", "delayed", "lost"]
    mentioned = [w for w in status_words if w in response_lower]

    if not mentioned:
        return True  # No status mentioned, that's fine

    # Every mentioned status should be valid
    return all(w in valid_statuses for w in mentioned)


print("\u2713 Code-based scorers defined:")
print("  mentions_order_id \u2014 checks response references the asked order ID")
print("  response_length \u2014 tracks response length (numeric metric)")
print("  no_hallucinated_status \u2014 ensures no invented order statuses")

✓ Code-based scorers defined:
  mentions_order_id — checks response references the asked order ID
  response_length — tracks response length (numeric metric)
  no_hallucinated_status — ensures no invented order statuses


## Custom LLM Judge with `make_judge()`

`make_judge()` lets you create an LLM-powered judge with custom instructions.
Unlike code-based scorers, these can assess **subjective quality** dimensions
that require language understanding.

The judge receives the trace context (inputs, outputs, tools called) and
returns a `yes`/`no` rating with a rationale explaining its decision.

In [0]:
"""Define a custom LLM judge using make_judge()."""

# A judge that evaluates tone and helpfulness.
# - `model` specifies which LLM serves as the judge (any Databricks serving endpoint)
# - `instructions` must reference {{ inputs }} and/or {{ outputs }} template variables
JUDGE_MODEL = "endpoints:/databricks-claude-sonnet-4"

tone_judge = make_judge(
    name="professional_tone",
    model=JUDGE_MODEL,
    instructions=(
        "You are evaluating the agent's response for professional customer service tone.\n\n"
        "User question: {{ inputs }}\n"
        "Agent response: {{ outputs }}\n\n"
        "Evaluate whether the response is:\n"
        "- Polite and empathetic\n"
        "- Clear and concise\n"
        "- Actionable (tells the customer what to expect next)\n\n"
        "Return 'yes' if the tone is professional and helpful, 'no' otherwise."
    ),
)

print(f"\u2713 Custom LLM judge defined:")
print(f"  Name: professional_tone")
print(f"  Judge model: {JUDGE_MODEL}")
print(f"  Evaluates: customer service tone quality")

✓ Custom LLM judge defined:
  Name: professional_tone
  Judge model: endpoints:/databricks-claude-sonnet-4
  Evaluates: customer service tone quality


## Run Evaluation: Custom + Built-in Scorers Together

The power of MLflow’s evaluation framework is that you can **mix and match** scorer types
in a single `evaluate()` call:
* Code-based scorers (fast, deterministic)
* Custom LLM judges (flexible, subjective)
* Built-in scorers (pre-configured, battle-tested)

In [0]:
"""Run evaluation combining all scorer types."""

MODEL_ENDPOINT = "databricks-llama-4-maverick"

JUDGE_MODEL = "endpoints:/databricks-llama-4-maverick"
tone_judge = make_judge(
    name="professional_tone",
    model=JUDGE_MODEL,
    instructions=(
        "You are evaluating the agent's response for professional customer service tone.\n\n"
        "User question: {{ inputs }}\n"
        "Agent response: {{ outputs }}\n\n"
        "Evaluate whether the response is:\n"
        "- Polite and empathetic\n"
        "- Clear and concise\n"
        "- Actionable (tells the customer what to expect next)\n\n"
        "Return 'yes' if the tone is professional and helpful, 'no' otherwise."
    ),
)

eval_data = pd.DataFrame([
    {"inputs": {"messages": [{"role": "user", "content": "What's the status of order ORD-1001?"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "Can you check on ORD-1002 for me?"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "Has order ORD-1003 been delivered yet?"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "I want to know about order ORD-1004"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "What does 'shipped' mean?"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "How long does standard shipping usually take?"}]}},
])


def predict_fn(messages: list) -> str:
    """Run the agent and return the final response."""
    user_msg = messages[0]["content"]
    return run_agent(user_msg)


print("Running combined evaluation (code + LLM + built-in scorers)...")
print("=" * 60)

eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        # Code-based (fast, deterministic)
        mentions_order_id,
        response_length,
        no_hallucinated_status,
        # Custom LLM judge
        tone_judge,
        # Built-in scorer
        ToolCallEfficiency(),
    ],
)

print("\n\u2713 Combined evaluation complete!")
print(f"\nAggregate Metrics:")
for name, value in sorted(eval_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {name}: {value:.3f}")
    else:
        print(f"  {name}: {value}")

2026/09/17 15:17:14 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Running combined evaluation (code + LLM + built-in scorers)...


Evaluating:   0%|          | 0/6 [Elapsed: 00:00, Remaining: ?] 

2026/09/17 15:17:20 WARNING mlflow.genai.judges.instructions_judge: The following parameters were provided but are not used by this judge's instructions: 'expectations'. The judge only uses template variables that appear in the instructions: {'outputs', 'inputs'}
/databricks/python/lib/python3.12/site-packages/mlflow/genai/judges/instructions_judge/__init__.py:562: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a future release. Please update your code to use the 'databricks' provider instead.
  return invoke_judge_model(



✓ Combined evaluation complete!

Aggregate Metrics:
  mentions_order_id/mean: 1.000
  no_hallucinated_status/mean: 1.000
  response_length/mean: 126.500
  tool_call_efficiency/mean: 1.000


In [0]:
"""Display per-row results."""
if eval_results.tables:
    results_df = list(eval_results.tables.values())[0]
    # Show compact view of scorer columns
    score_cols = [c for c in results_df.columns if "/value" in c or "/mean" in c or c == "trace_id"]
    if score_cols:
        display(results_df[score_cols])
    else:
        display(results_df)

trace_id,mentions_order_id/value,professional_tone/value,tool_call_efficiency/value,response_length/value,no_hallucinated_status/value
tr-58eb0694cac5acd09cd4048662d6d69a,true,null,yes,61.0,true
tr-adc8509c1afe3afe1ba9d0391649ce3b,true,null,yes,151.0,true
tr-e0a72dba90110dfcaabfb90ec63d9b11,true,null,yes,79.0,true
tr-2c781a0055b72c9ab9a13778fcca75e5,true,null,yes,60.0,true
tr-acc74f550bdca2c1fe5b4e411bcc9816,true,null,yes,324.0,true
tr-6f0ba1b33f37eb4bbbb4770beb114c99,true,null,yes,84.0,true


## Summary

What you learned:
* `@scorer` creates **code-based scorers** — fast, deterministic, zero-cost
* `make_judge()` creates **custom LLM judges** — flexible, domain-specific, subjective
* Both integrate seamlessly with `mlflow.genai.evaluate()` alongside built-in scorers
* Mix scorer types freely: code + LLM + built-in in one `evaluate()` call

**When to use which:**

| Scorer Type | Use When | Cost | Latency |
| --- | --- | --- | --- |
| `@scorer` (code) | Rules expressible in Python (regex, format checks) | Free | \~0ms |
| `make_judge()` (LLM) | Subjective quality (tone, helpfulness, accuracy) | LLM call | \~1-3s |
| Built-in (`ToolCallEfficiency`, etc.) | Standard dimensions already covered by MLflow | LLM call | \~1-3s |